In [ ]:
# --- PocketLM setup (works in Colab and locally) ---
# In Colab the package AND data/ must be present, so we clone the repo (Colab opens
# only the single .ipynb, not the repo). In CI/local the package is already installed
# from the checkout under test, so we skip the clone and exercise that code.
import os

try:
    import pocketlm  # already installed (local/CI) -> use the code under test
except ImportError:
    import subprocess
    import sys

    if not os.path.isdir("pocketlm-repo"):
        subprocess.check_call(
            ["git", "clone", "--depth", "1",
             "https://github.com/ni5h4nt/pocketlm", "pocketlm-repo"])
    os.chdir("pocketlm-repo")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", "."])
    import pocketlm  # noqa: F811

import torch

# nbmake runs a notebook from its own directory; anchor the cwd at the repo root
# (derived from the installed package) so CORPUS_PATH resolves in CI, locally, and
# in Colab (where the except-branch already chdir'd into the clone).
os.chdir(os.path.dirname(os.path.dirname(os.path.abspath(pocketlm.__file__))))

DEVICE = "micro" if os.environ.get("POCKETLM_CI") else ("cuda" if torch.cuda.is_available() else "cpu")
CORPUS_PATH = "data/corpus.txt"   # repo-root-relative; valid after the chdir above
print("device:", DEVICE, "(timing is guidance, not a contract)")

# l2.3 Vocabulary design

Vocabulary size is a budget. A bigger vocab learns more merges, so it encodes
text in fewer tokens, but every token costs an embedding row and a logit. The
engineering job is choosing the trade.

In [ ]:
from pocketlm import BPETokenizer

corpus = open(CORPUS_PATH, encoding="utf-8").read()
sample = corpus[:20000]
small = BPETokenizer().train(sample, 300)
big = BPETokenizer().train(sample, 600)
probe = corpus[20000:21000]
print("vocab 300 ->", len(small.encode(probe)), "tokens")
print("vocab 600 ->", len(big.encode(probe)), "tokens")

Greedy BPE is nested: the 600-token vocab contains every merge the 300-token
vocab learned, plus more, so it can only encode in fewer-or-equal tokens.

In [ ]:
assert len(big.encode(probe)) <= len(small.encode(probe))